**Install**

In [1]:
!pip install transformers

**Set-up**

In [2]:
from transformers import pipeline

llm = pipeline("text-generation", model="distilgpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/353M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

**Data**

In [3]:
RESUMES = {
    "Strong Candidate": """
    Data Scientist with 5 years experience.
    Skills: Python, Machine Learning, Deep Learning, NLP, SQL.
    Tools: TensorFlow, PyTorch, Pandas.
    """,

    "Average Candidate": """
    Data Analyst with 2 years experience.
    Skills: Python, Excel, SQL.
    Tools: Pandas, Power BI.
    """,

    "Weak Candidate": """
    Fresher with basic knowledge.
    Skills: Excel.
    """
}

JOB_DESCRIPTION = """
Looking for Data Scientist.
Required Skills: Python, Machine Learning, NLP, SQL.
Experience: 3+ years.
"""

**Pipeline Function**

In [4]:
def generate(prompt):
    output = llm(
        prompt,
        max_new_tokens=60,
        do_sample=False
    )
    return output[0]['generated_text'].replace(prompt, "").strip()

In [5]:
from transformers import pipeline

# Load model
llm = pipeline("text-generation", model="google/flan-t5-base")

# Define generate function
def generate(prompt):
    return llm(prompt, max_length=200)[0]['generated_text']

JOB = """
Looking for Data Scientist.
Required Skills: Python, Machine Learning, NLP, SQL.
Experience: 3+ years.
"""

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'LlamaForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DiffLl

**Run Pipeline**

In [12]:
import re

def extract_info(resume):
    skills = []
    skill_keywords = ["python", "machine learning", "nlp", "sql", "excel", "deep learning"]

    for skill in skill_keywords:
        if skill.lower() in resume.lower():
            skills.append(skill)

    # extract experience
    exp_match = re.search(r"(\d+)\s*years", resume.lower())
    experience = int(exp_match.group(1)) if exp_match else 0

    return {
        "skills": skills,
        "experience": experience
    }


def match_profile(profile, job_skills):
    matched = [s for s in profile["skills"] if s in job_skills]
    missing = [s for s in job_skills if s not in profile["skills"]]

    return {
        "matched": matched,
        "missing": missing
    }


def score_candidate(match, experience):
    skill_score = (len(match["matched"]) / 4) * 70
    exp_score = min(experience / 5, 1) * 40

    score = int(skill_score + exp_score)
    score = min(score, 100)   # 👈 fix

    if score >= 75:
        level = "Strong"
    elif score >= 40:
        level = "Average"
    else:
        level = "Weak"

    return score, level

def explain(score, match):
    return (
        f"The candidate matches {len(match['matched'])} required skills "
        f"and lacks {len(match['missing'])}. "
        f"The score reflects both skill alignment and experience level."
    )

# JOB SKILLS
job_skills = ["python", "machine learning", "nlp", "sql"]


# RUN PIPELINE
for name, resume in RESUMES.items():
    print("\n==============================")
    print(f"Candidate: {name}")

    extracted = extract_info(resume)
    print("Extracted:", extracted)

    matched = match_profile(extracted, job_skills)
    print("Match:", matched)

    score, level = score_candidate(matched, extracted["experience"])
    print(f"Score: {score}/100 | Level: {level}")

    explanation = explain(score, matched)
    print("Explanation:", explanation)


Candidate: Strong Candidate
Extracted: {'skills': ['python', 'machine learning', 'nlp', 'sql', 'deep learning'], 'experience': 5}
Match: {'matched': ['python', 'machine learning', 'nlp', 'sql'], 'missing': []}
Score: 100/100 | Level: Strong
Explanation: The candidate matches 4 required skills and lacks 0. The score reflects both skill alignment and experience level.

Candidate: Average Candidate
Extracted: {'skills': ['python', 'sql', 'excel'], 'experience': 2}
Match: {'matched': ['python', 'sql'], 'missing': ['machine learning', 'nlp']}
Score: 51/100 | Level: Average
Explanation: The candidate matches 2 required skills and lacks 2. The score reflects both skill alignment and experience level.

Candidate: Weak Candidate
Extracted: {'skills': ['excel'], 'experience': 0}
Match: {'matched': [], 'missing': ['python', 'machine learning', 'nlp', 'sql']}
Score: 0/100 | Level: Weak
Explanation: The candidate matches 0 required skills and lacks 4. The score reflects both skill alignment and ex